# Resume Builder
## Goal
Create a tailored resume based on a complete "master resume" doc. The master resume contains is typically very long as it covers all interesting / relevant info for a person's career. Use example job descriptions as guidelines for the tailored resume.
## Workflow
<ol>
<li>Load & normalize master resume</li>
<li>Build canonical resume xml</li>
<li>Load contracts</li>
<li>Generate targeted resume contentin XML</li>
<li>Choose layout config, YAML</li>
<li>Render as PDF</li>
<li>Inspect resume diagnostics</li>
<li>Revise content or layout</li>
</ol>



## 1. Load & normalize master resume

In [ ]:
from docx import Document
from pathlib import Path
import re
VERBOSE = True
ARTIFACT_DIR = Path("artifacts")


doc_path = Path("data") / "Master Resume.docx"
doc = Document(doc_path)

# Separate entire doc into paragraphs and remove empty paragraphs
paragraphs = []

for p in doc.paragraphs:
    raw = p.text.strip()

    if not raw:
        continue

    for line in raw.splitlines():
        line = line.strip()
        if line:
            paragraphs.append(line)

### Normalize DOCX Blocks

Google Docs exported to DOCX does not preserve useful resume structure for this document.
Most lines appear with the style `Normal`, so we should not rely on Word styles to detect sections, headings, or bullets.

Instead, this step converts DOCX paragraphs into normalized text blocks:

DOCX paragraphs
→ split embedded line breaks
→ trim whitespace
→ preserve available metadata
→ stop before Career Archive

Later, if we load resumes from other formats, each loader should produce the same `normalized_blocks` structure.

In [2]:
# Convert paragraphs into normalized blocks which preserve formatting metadata
normalized_blocks = []

for p in doc.paragraphs:
    raw = p.text.strip()

    if not raw:
        continue

    for line in raw.splitlines():
        line = line.strip()

        if line:
            normalized_blocks.append({
                "text": line,
                "source_style": p.style.name,
            })

In [3]:
# Remove archive section and preview resume. Archive separator is a line with a specific text defined.
# Do text cleanup to get rid of extra spaces, newlines, and colons. Normalize text to lowercase for comparison. 
# Observation: DOCX paragraphs are not guaranteed to equal logical resume lines.

ARCHIVE_MARKERS = {
    "career archive",
    "archive",
    "career history archive",
}

def normalize_header(text: str) -> str:
    return " ".join(text.strip().lower().replace("\xa0", " ").split()).rstrip(":")


def split_active_and_archive(blocks):
    active = []
    archive = []
    in_archive = False

    for block in blocks:
        if normalize_header(block["text"]) in ARCHIVE_MARKERS:
            in_archive = True
            archive.append(block)
            continue

        if in_archive:
            archive.append(block)
        else:
            active.append(block)

    return active, archive

active_blocks, archive_blocks = split_active_and_archive(normalized_blocks)
if VERBOSE:
    print(f"Total raw paragraphs: {len(normalized_blocks)}")
    print(f"Active resume paragraphs: {len(active_blocks)}")
    print(f"Archive paragraphs: {len(archive_blocks)}")
    for p in active_blocks:
        print(p)

Total raw paragraphs: 260
Active resume paragraphs: 97
Archive paragraphs: 163
{'text': 'CONTACT', 'source_style': 'Heading 1'}
{'text': 'Location: Portland, OR', 'source_style': 'normal'}
{'text': 'Email: douglasgdaly@gmail.com', 'source_style': 'normal'}
{'text': 'Phone: (650) 586-9720', 'source_style': 'normal'}
{'text': 'LinkedIn: linkedin.com/in/douglasdaly', 'source_style': 'normal'}
{'text': 'GitHub: github.com/dougdaly', 'source_style': 'normal'}
{'text': 'SUMMARY', 'source_style': 'normal'}
{'text': 'Principal AI Engineer and Technical Leader with 20+ years of experience building systems that help organizations understand what is happening, why it is happening, and what to do next. Expertise spans operational intelligence, decision-support systems, attribution analytics, observability, graph analytics, and AI-enabled business systems.', 'source_style': 'normal'}
{'text': 'Designed and delivered solutions ranging from a patented radar resource-management system and enterprise-s

## 2. Build Canonical Resume
MASTER_SECTIONS are from the master resume doc. Modify per doc layout.

In [4]:
# Define parameters for master resume. Define sections and type of parsers.
MASTER_SECTION_HEADERS = {
    "contact",
    "summary",
    "core technologies",
    "professional experience",
    "education",
    "certifications",
    "projects",
}

SECTION_ALIASES = {
    "contact": "Contact",
    "summary": "Summary",
    "core technologies": "Core Technologies",
    "education": "Education",
    "certifications": "Certifications",
    "professional experience": "Professional Experience",
    "projects": "Projects",
    "selected projects": "Projects",
    "patents & recognition": "Certifications"
}

MASTER_SECTION_RULES = {
    "Contact": {
        "block_parser": "text_only",
    },
    "Summary": {
        "block_parser": "text_only",
    },
    "Core Technologies": {
        "block_parser": "colon_heading_text",
    },
    "Professional Experience": {
        "block_parser": "item_section",
    },
    "Education": {
        "block_parser": "text_only",
    },
    "Certifications": {
        "block_parser": "text_only",
    },
    "Projects": {
        "block_parser": "item_section",
    },
}

CONTENT_DELIMITERS = ("•", "-", "*")
ITEM_HEADING_DELIMITER = "|"


In [5]:
# helpers
def is_content_line(text: str) -> bool:
    return text.strip().startswith(CONTENT_DELIMITERS)


def clean_content_line(text: str) -> str:
    text = text.strip()
    for delimiter in CONTENT_DELIMITERS:
        if text.startswith(delimiter):
            return text[len(delimiter):].strip()
    return text


def is_item_heading(text: str) -> bool:
    text = text.strip()
    return ITEM_HEADING_DELIMITER in text and any(ch.isdigit() for ch in text)

In [6]:
# Define supported parsers for section types
def parse_text_only_section(section):
    return {
        "name": section["name"],
        "blocks": [
            {
                "kind": "text",
                "text": block["text"],
            }
            for block in section["blocks"]
        ]
    }

def parse_colon_heading_text_section(section: dict) -> dict:
    parsed_blocks = []

    for block in section["blocks"]:
        text = block["text"]

        if ":" in text:
            heading, body = text.split(":", 1)

            parsed_blocks.append({
                "kind": "block",
                "heading": heading.strip(),
                "text": body.strip(),
            })
        else:
            parsed_blocks.append({
                "kind": "text",
                "text": text,
            })

    return {
        "name": section["name"],
        "blocks": parsed_blocks,
    }

def parse_item_section(section: dict) -> dict:
    parsed_blocks = []
    current_item = None
    current_group = None

    for block in section["blocks"]:
        text = block["text"].strip()
        if not text:
            continue

        if is_item_heading(text):
            current_item = {
                "kind": "item",
                "heading": text,
                "groups": [],
            }
            parsed_blocks.append(current_item)
            current_group = None
            continue

        if current_item is None:
            parsed_blocks.append({"kind": "text", "text": clean_content_line(text)})
            continue

        if is_content_line(text):
            if current_group is None:
                current_group = {
                    "subheading": None,
                    "content": [],
                }
                current_item["groups"].append(current_group)

            current_group["content"].append({
                "kind": "bullet",
                "text": clean_content_line(text),
            })
            continue

        # Non-heading, non-bullet inside an item = subgroup heading
        current_group = {
            "subheading": text,
            "content": [],
        }
        current_item["groups"].append(current_group)

    return {
        "name": section["name"],
        "blocks": parsed_blocks,
    }

PARSERS = {
    "text_only": parse_text_only_section,
    "colon_heading_text": parse_colon_heading_text_section,
    "item_section": parse_item_section,
}


In [7]:

# Convert active blocks into structured sections based on formatting and text patterns. 
def block_kind(block: dict) -> str:
    text_norm = normalize_header(block["text"])

    if text_norm in SECTION_ALIASES:
        return "section_header"

    if block["source_style"].lower().startswith("heading"):
        return "heading"

    return "text"


annotated_blocks = []
for block in active_blocks:
    annotated_blocks.append({
        **block,
        "kind": block_kind(block),
    })

if VERBOSE:
    for b in annotated_blocks[:30]:
        print(b)

{'text': 'CONTACT', 'source_style': 'Heading 1', 'kind': 'section_header'}
{'text': 'Location: Portland, OR', 'source_style': 'normal', 'kind': 'text'}
{'text': 'Email: douglasgdaly@gmail.com', 'source_style': 'normal', 'kind': 'text'}
{'text': 'Phone: (650) 586-9720', 'source_style': 'normal', 'kind': 'text'}
{'text': 'LinkedIn: linkedin.com/in/douglasdaly', 'source_style': 'normal', 'kind': 'text'}
{'text': 'GitHub: github.com/dougdaly', 'source_style': 'normal', 'kind': 'text'}
{'text': 'SUMMARY', 'source_style': 'normal', 'kind': 'section_header'}
{'text': 'Principal AI Engineer and Technical Leader with 20+ years of experience building systems that help organizations understand what is happening, why it is happening, and what to do next. Expertise spans operational intelligence, decision-support systems, attribution analytics, observability, graph analytics, and AI-enabled business systems.', 'source_style': 'normal', 'kind': 'text'}
{'text': 'Designed and delivered solutions rang

In [8]:
# Build resume structure as a collection of sections.
resume = {
    "sections": []
}

current_section = None
for block in annotated_blocks:
    if block["kind"] == "section_header":
        normalized = normalize_header(block["text"])
        if normalized in SECTION_ALIASES:
            section_name = SECTION_ALIASES[normalized]
        current_section = {
            "name": SECTION_ALIASES[normalized],
            "blocks": []
        }
        resume["sections"].append(current_section)
        continue

    if current_section is None:
        current_section = {
            "name": "Uncategorized",
            "blocks": []
        }
        resume["sections"].append(current_section)

    current_section["blocks"].append(block)

In [9]:
parsed_resume = {
    "sections": []
}

for section in resume["sections"]:
    print(section['name'])
    rule = MASTER_SECTION_RULES.get(
        SECTION_ALIASES[section["name"].strip().lower()],
        {"block_parser": "text_only"}
    )

    parser_name = rule["block_parser"]
    parser_fn = PARSERS[parser_name]
    parsed_section = parser_fn(section)
    parsed_resume["sections"].append(parsed_section)

Contact
Summary
Core Technologies
Professional Experience
Projects
Education
Certifications


### Save parsed resume as JSON object

In [ ]:
import json
resume_metadata = {
    "schema_version": "1.0",
    "source_file": "Master Resume.docx",
    "section_count": len(parsed_resume["sections"]),
}

master_resume = {
    "metadata": resume_metadata,
    "resume": parsed_resume,
}

with open(ARTIFACT_DIR / "master_resume.json", "w") as f:
    json.dump(master_resume, f, indent=2)

# Phase 1 Complete

Master Resume Successfully Converted To Canonical Resume Format

Artifact:
    master_resume.json

Next Phase:
    Resume Targeting